# Laya Local + CarRacing — zero-shot System-1 driving experiment

[Open in Colab](https://colab.research.google.com/github/vtavakkoli/simple-jev/blob/main/notebooks/Laya_CarRacing_Local_Colab.ipynb)

This notebook runs convaiinnovations/laya locally and lets it choose the actual steering/gas/brake maneuver for Gymnasium CarRacing-v3.

Laya is not a vision model, so the 96x96 RGB observation is first converted into compact road geometry: near/mid/far road-center offsets, curvature, visibility, and a categorical turn hint. Basic simulator speed telemetry is added. Laya then makes one 11-way typed choice, mapped directly to the continuous CarRacing action.

- No TypeSafe/Jev API and no external LLM API.
- Hugging Face weights are downloaded once; inference then runs in the Colab/local runtime.
- Default checkpoint: convaiinnovations/laya, English 421M.
- This is a zero-shot experiment, not a trained racing policy.
- Laya confidence is logged but is not treated as a calibrated driving-safety probability.


In [ ]:
#@title 1. Install dependencies
!apt-get -qq update
!apt-get -qq install -y swig > /dev/null
!pip -q install "gymnasium[box2d]==1.3.0" laya imageio imageio-ffmpeg opencv-python-headless matplotlib pandas


In [ ]:
#@title 2. Imports, configuration, and local Laya load
import os
os.environ["USE_TF"] = "0"

import time, json
from collections import Counter, deque
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from IPython.display import Video, display
from laya import Router

MODEL_ID = "convaiinnovations/laya"
SEED = 0
MAX_STEPS = 1000
DECISION_EVERY = 4
VIDEO_EVERY = 2
POLICY = "laya"  # change to "vision_baseline" only for comparison
LOST_ROAD_BRAKE_AFTER = 3

OUTPUT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
VIDEO_PATH = str(OUTPUT_DIR / "laya_carracing_local.mp4")
LOG_PATH = str(OUTPUT_DIR / "laya_carracing_local_log.json")
CSV_PATH = str(OUTPUT_DIR / "laya_carracing_local_decisions.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

router = Router(device=DEVICE, max_loaded=1)
agent = router.load("english")
if hasattr(agent, "cfg") and isinstance(agent.cfg, dict):
    agent.cfg["max_len"] = min(int(agent.cfg.get("max_len", 512)), 512)
    agent.cfg["head_max_len"] = min(256, int(agent.cfg["max_len"]) - 160)

print("✓ Laya loaded locally:", MODEL_ID)
print("✓ no API key is used")


In [ ]:
#@title 3. RGB road perception -> compact state
IMG_W = IMG_H = 96
CX = (IMG_W - 1) / 2.0
SCAN_YS = [69, 60, 51, 42, 33, 25]

def _runs(xs):
    if len(xs) == 0:
        return []
    out, start, prev = [], int(xs[0]), int(xs[0])
    for x in xs[1:]:
        x = int(x)
        if x != prev + 1:
            out.append((start, prev))
            start = x
        prev = x
    out.append((start, prev))
    return out

def build_road_mask(obs):
    rgb = np.asarray(obs, dtype=np.uint8).astype(np.int16)
    hi, lo = rgb.max(axis=2), rgb.min(axis=2)
    road = (
        ((hi - lo) < 34)
        & (rgb[:, :, 0] > 55) & (rgb[:, :, 0] < 190)
        & (rgb[:, :, 1] > 55) & (rgb[:, :, 1] < 190)
        & (rgb[:, :, 2] > 55) & (rgb[:, :, 2] < 190)
    )
    road[82:, :] = False
    m = road.astype(np.uint8) * 255
    k = np.ones((3, 3), np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, k)
    return m > 0

def extract_road_features(obs):
    mask = build_road_mask(obs)
    centers, widths, points = [], [], []
    target_x = CX

    for y in SCAN_YS:
        density = mask[max(0,y-2):min(IMG_H,y+3)].mean(axis=0)
        candidates = []
        for a, b in _runs(np.flatnonzero(density >= 0.40)):
            w = b - a + 1
            if w >= 4:
                c = 0.5 * (a + b)
                candidates.append((abs(c-target_x), -w, c, w))
        if not candidates:
            centers.append(np.nan); widths.append(0.0); points.append(None)
            continue
        _, _, c, w = min(candidates)
        target_x = c
        centers.append(float(c)); widths.append(float(w)); points.append((int(round(c)), y))

    valid = np.isfinite(centers)
    nvalid = int(valid.sum())
    if nvalid:
        filled = np.asarray(centers, dtype=float)
        vi = np.flatnonzero(np.isfinite(filled))
        for i in range(len(filled)):
            if not np.isfinite(filled[i]):
                j = vi[np.argmin(np.abs(vi-i))]
                filled[i] = filled[j]
    else:
        filled = np.full(len(SCAN_YS), CX)

    offs = (filled - CX) / CX
    near, mid, far = float(np.mean(offs[:2])), float(np.mean(offs[2:4])), float(np.mean(offs[4:]))
    curvature = float(np.clip(far-near, -1, 1))
    vw = [w/IMG_W for w in widths if w > 0]
    mean_width = float(np.mean(vw)) if vw else 0.0
    conf = float(np.clip(0.78*(nvalid/len(SCAN_YS)) + 0.22*min(mean_width/0.35,1), 0, 1))
    path_error = float(np.clip(0.50*far + 0.25*mid + 0.45*curvature, -1, 1))

    if conf < 0.28: hint = "unknown"
    elif path_error < -0.34: hint = "sharp_left"
    elif path_error < -0.11: hint = "left"
    elif path_error > 0.34: hint = "sharp_right"
    elif path_error > 0.11: hint = "right"
    else: hint = "straight"

    visibility = "lost" if conf < 0.28 else "partial" if conf < 0.60 else "good"
    return dict(
        near_offset=near, mid_offset=mid, far_offset=far, curvature=curvature,
        path_error=path_error, turn_hint=hint, visibility=visibility,
        road_confidence=conf, valid_scanlines=nvalid, mean_road_width=mean_width,
        _mask=mask, _points=points
    )

def vehicle_telemetry(env):
    try:
        h = env.unwrapped.car.hull
        vx, vy = h.linearVelocity
        speed = float(np.hypot(vx, vy))
        omega = float(h.angularVelocity)
    except Exception:
        speed, omega = 0.0, 0.0
    state = "stopped" if speed < 4 else "slow" if speed < 14 else "cruise" if speed < 28 else "fast" if speed < 40 else "very_fast"
    return dict(speed=speed, speed_state=state, angular_velocity=omega)

def show_perception(obs):
    f = extract_road_features(obs)
    overlay = np.asarray(obs).copy()
    for p in f["_points"]:
        if p is not None: cv2.circle(overlay, p, 2, (255,255,0), -1)
    cv2.line(overlay, (int(CX),18), (int(CX),78), (255,0,255), 1)
    plt.figure(figsize=(6,6)); plt.imshow(overlay)
    plt.title(f"hint={f['turn_hint']} conf={f['road_confidence']:.2f} near={f['near_offset']:+.2f} far={f['far_offset']:+.2f}")
    plt.axis("off"); plt.show()
    plt.figure(figsize=(6,6)); plt.imshow(f["_mask"], cmap="gray")
    plt.title("road mask"); plt.axis("off"); plt.show()
    return {k:v for k,v in f.items() if not k.startswith("_")}

dbg = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array")
dbg_obs, _ = dbg.reset(seed=SEED)
print(json.dumps(show_perception(dbg_obs), indent=2))
dbg.close()


In [ ]:
#@title 4. Laya action schema + smoke test
MANEUVER_ACTIONS = {
    "hard_left":      np.array([-0.90,0.28,0.00], np.float32),
    "left":           np.array([-0.58,0.40,0.00], np.float32),
    "soft_left":      np.array([-0.28,0.55,0.00], np.float32),
    "straight":       np.array([ 0.00,0.65,0.00], np.float32),
    "soft_right":     np.array([ 0.28,0.55,0.00], np.float32),
    "right":          np.array([ 0.58,0.40,0.00], np.float32),
    "hard_right":     np.array([ 0.90,0.28,0.00], np.float32),
    "coast":          np.array([ 0.00,0.00,0.00], np.float32),
    "brake_left":     np.array([-0.42,0.00,0.32], np.float32),
    "brake_straight": np.array([ 0.00,0.00,0.42], np.float32),
    "brake_right":    np.array([ 0.42,0.00,0.32], np.float32),
}
CRITERIA = {
    "hard_left":"sharp left road ahead; strong left steering, limited throttle",
    "left":"clear left bend; medium left steering with moderate throttle",
    "soft_left":"small left correction; gentle left steering and good throttle",
    "straight":"road centered and straight; accelerate smoothly",
    "soft_right":"small right correction; gentle right steering and good throttle",
    "right":"clear right bend; medium right steering with moderate throttle",
    "hard_right":"sharp right road ahead; strong right steering, limited throttle",
    "coast":"temporarily reduce acceleration when uncertain or unstable",
    "brake_left":"slow down while correcting left on a difficult left turn",
    "brake_straight":"slow down without adding steering",
    "brake_right":"slow down while correcting right on a difficult right turn",
}
QUESTIONS = {"maneuver":{
    "type":"choice",
    "instructions":(
        "Choose the next maneuver for Gymnasium CarRacing-v3. Negative road offsets and curvature mean left; positive mean right. "
        "Follow turn_hint unless numeric geometry strongly contradicts it. Keep the car near road center. Prefer acceleration when "
        "visibility is good and speed is stopped/slow/cruise. Reduce throttle on hard bends or high speed. Brake only when fast, "
        "very_fast, the road is difficult, or recovery is needed. Never choose left for right geometry or right for left geometry."
    ),
    "criteria":CRITERIA
}}

def compact_state(f,t,step,total_reward,recent,previous):
    rr = list(recent)
    return {
        "task":"drive around visible asphalt track without leaving the road",
        "step":int(step),
        "road":{
            "visibility":f["visibility"], "confidence":round(f["road_confidence"],3),
            "turn_hint":f["turn_hint"], "near_offset":round(f["near_offset"],3),
            "mid_offset":round(f["mid_offset"],3), "far_offset":round(f["far_offset"],3),
            "curvature":round(f["curvature"],3)
        },
        "vehicle":{"speed_state":t["speed_state"],"speed":round(t["speed"],2),"angular_velocity":round(t["angular_velocity"],3)},
        "history":{"previous_maneuver":previous,"recent_reward_mean":round(float(np.mean(rr)) if rr else 0,3),"total_reward":round(float(total_reward),2)}
    }

def _field(x,k,d=None):
    return x.get(k,d) if isinstance(x,dict) else getattr(x,k,d)

def laya_choose(state):
    t0 = time.perf_counter()
    r = router.predict(state, QUESTIONS, model="english")
    ms = (time.perf_counter()-t0)*1000
    a = _field(r,"answers",{})["maneuver"]
    choice = _field(a,"choice")
    conf = float(_field(a,"confidence",np.nan))
    probs = _field(a,"probabilities",{}) or {}
    if choice not in MANEUVER_ACTIONS:
        raise RuntimeError(f"unknown Laya maneuver: {choice!r}")
    return {"maneuver":choice,"confidence":conf,"probabilities":{str(k):float(v) for k,v in dict(probs).items()},"latency_ms":ms}

def baseline(f,t):
    h,s = f["turn_hint"], t["speed_state"]
    if h=="sharp_left": return "brake_left" if s in ("fast","very_fast") else "hard_left"
    if h=="left": return "left"
    if h=="sharp_right": return "brake_right" if s in ("fast","very_fast") else "hard_right"
    if h=="right": return "right"
    if h=="unknown": return "coast"
    return "brake_straight" if s=="very_fast" else "straight"

def smooth(prev,target):
    prev,target=np.asarray(prev,np.float32),np.asarray(target,np.float32)
    o=target.copy()
    o[0]=0.58*prev[0]+0.42*target[0]
    o[1]=0.30*prev[1]+0.70*target[1]
    o[2]=0.30*prev[2]+0.70*target[2]
    if o[1]>0.05 and o[2]>0.05:
        if o[1]>=o[2]: o[2]=0
        else: o[1]=0
    return np.clip(o,[-1,0,0],[1,1,1]).astype(np.float32)

def synthetic(hint,far,curve):
    return {
        "task":"drive around visible asphalt track without leaving the road","step":0,
        "road":{"visibility":"good","confidence":0.95,"turn_hint":hint,"near_offset":0.0,"mid_offset":round(far*0.55,3),"far_offset":far,"curvature":curve},
        "vehicle":{"speed_state":"cruise","speed":20.0,"angular_velocity":0.0},
        "history":{"previous_maneuver":"straight","recent_reward_mean":0.0,"total_reward":0.0}
    }

for name,st in [("LEFT",synthetic("sharp_left",-0.58,-0.42)),("STRAIGHT",synthetic("straight",0.01,0.01)),("RIGHT",synthetic("sharp_right",0.58,0.42))]:
    d=laya_choose(st)
    top=sorted(d["probabilities"].items(),key=lambda kv:kv[1],reverse=True)[:4]
    print(f"{name:8s} -> {d['maneuver']:>14s} | conf={d['confidence']:.3f} | {d['latency_ms']:.1f} ms | top={top}")


If the smoke test fails to separate left / straight / right, that is already useful evidence that the generic checkpoint does not transfer cleanly zero-shot to numerical racing control. The episode below still measures the real behavior; it does not silently replace Laya with a trained RL policy.


In [ ]:
#@title 5. Run the episode
env = gym.make("CarRacing-v3", continuous=True, render_mode="rgb_array", max_episode_steps=MAX_STEPS)
obs,_ = env.reset(seed=SEED)

total_reward=0.0
recent=deque(maxlen=25)
maneuver="straight"
raw_action=MANEUVER_ACTIONS[maneuver].copy()
action=np.zeros(3,np.float32)
decisions=[]
reward_history=[]
speed_history=[]
laya_failures=sensor_fallbacks=lost_streak=0
terminated=truncated=False

writer=imageio.get_writer(VIDEO_PATH,format="FFMPEG",mode="I",fps=50.0/VIDEO_EVERY,codec="libx264",pixelformat="yuv420p",macro_block_size=1)

try:
    for step in range(MAX_STEPS):
        f=extract_road_features(obs)
        t=vehicle_telemetry(env)

        if step % DECISION_EVERY == 0:
            lost_streak = lost_streak+1 if f["visibility"]=="lost" else 0
            state=compact_state(f,t,step,total_reward,recent,maneuver)
            override=None
            conf=lat=np.nan
            probs={}
            raw_laya=None

            if POLICY=="laya":
                try:
                    d=laya_choose(state)
                    raw_laya=maneuver=d["maneuver"]
                    conf,probs,lat=d["confidence"],d["probabilities"],d["latency_ms"]
                except Exception as e:
                    laya_failures+=1
                    maneuver=baseline(f,t)
                    override=f"laya_error:{type(e).__name__}"
                    print("Laya error -> baseline:",repr(e))
                if lost_streak>=LOST_ROAD_BRAKE_AFTER:
                    maneuver="brake_straight"
                    sensor_fallbacks+=1
                    override="road_sensor_lost"
            else:
                maneuver=baseline(f,t)
                override="vision_baseline"

            raw_action=MANEUVER_ACTIONS[maneuver].copy()
            rec={
                "step":step,"raw_laya_maneuver":raw_laya,"applied_maneuver":maneuver,"override_reason":override,
                "confidence":None if not np.isfinite(conf) else float(conf),
                "latency_ms":None if not np.isfinite(lat) else float(lat),
                "probabilities":probs,"turn_hint":f["turn_hint"],"visibility":f["visibility"],
                "road_confidence":float(f["road_confidence"]),"near_offset":float(f["near_offset"]),
                "mid_offset":float(f["mid_offset"]),"far_offset":float(f["far_offset"]),
                "curvature":float(f["curvature"]),"speed":float(t["speed"]),"speed_state":t["speed_state"],
                "total_reward":float(total_reward)
            }
            decisions.append(rec)
            if len(decisions)<=12 or len(decisions)%10==0:
                c="n/a" if rec["confidence"] is None else f"{rec['confidence']:.3f}"
                l="n/a" if rec["latency_ms"] is None else f"{rec['latency_ms']:.1f}ms"
                print(f"decision {len(decisions):03d} step {step:04d} | vision={f['turn_hint']:>11s} | Laya={str(raw_laya):>14s} | apply={maneuver:>14s} | conf={c} | {l} | speed={t['speed']:.1f} | reward={total_reward:.1f}")

        action=smooth(action,raw_action)
        obs,reward,terminated,truncated,info=env.step(action)
        total_reward+=float(reward)
        recent.append(float(reward))
        reward_history.append(total_reward)
        speed_history.append(vehicle_telemetry(env)["speed"])
        if step%VIDEO_EVERY==0:
            writer.append_data(env.render())
        if terminated or truncated:
            print(f"episode ended at step {step+1}: terminated={terminated} truncated={truncated}")
            break
finally:
    writer.close()
    env.close()

latencies=[d["latency_ms"] for d in decisions if d["latency_ms"] is not None]
confidences=[d["confidence"] for d in decisions if d["confidence"] is not None]
summary={
    "model":MODEL_ID,"device":DEVICE,"policy":POLICY,"seed":SEED,"steps":len(reward_history),
    "total_reward":float(total_reward),"decisions":len(decisions),"laya_failures":laya_failures,
    "sensor_fallbacks":sensor_fallbacks,"terminated":bool(terminated),"truncated":bool(truncated),
    "mean_laya_latency_ms":float(np.mean(latencies)) if latencies else None,
    "mean_laya_confidence":float(np.mean(confidences)) if confidences else None,
    "maneuver_counts":dict(Counter(d["applied_maneuver"] for d in decisions))
}
Path(LOG_PATH).write_text(json.dumps({"summary":summary,"decisions":decisions},indent=2),encoding="utf-8")
rows=[{k:v for k,v in d.items() if k!="probabilities"} for d in decisions]
pd.DataFrame(rows).to_csv(CSV_PATH,index=False)
print("\n=== RESULT ===")
print(json.dumps(summary,indent=2))


In [ ]:
#@title 6. Video and diagnostics
display(Video(VIDEO_PATH,embed=True,html_attributes="controls loop"))

plt.figure(figsize=(12,4)); plt.plot(reward_history)
plt.xlabel("step"); plt.ylabel("cumulative reward"); plt.title("CarRacing reward")
plt.grid(True,alpha=0.25); plt.show()

plt.figure(figsize=(12,4)); plt.plot(speed_history)
plt.xlabel("step"); plt.ylabel("speed"); plt.title("Vehicle speed")
plt.grid(True,alpha=0.25); plt.show()

if decisions:
    xs=[d["step"] for d in decisions]
    conf=[np.nan if d["confidence"] is None else d["confidence"] for d in decisions]
    lat=[np.nan if d["latency_ms"] is None else d["latency_ms"] for d in decisions]

    plt.figure(figsize=(12,4)); plt.plot(xs,conf,marker=".",linewidth=1)
    plt.xlabel("step"); plt.ylabel("confidence"); plt.ylim(-0.02,1.02)
    plt.title("Laya confidence — diagnostic, not safety calibration")
    plt.grid(True,alpha=0.25); plt.show()

    plt.figure(figsize=(12,4)); plt.plot(xs,lat,marker=".",linewidth=1)
    plt.xlabel("step"); plt.ylabel("local inference latency (ms)")
    plt.title("Laya local decision latency")
    plt.grid(True,alpha=0.25); plt.show()

pd.DataFrame(rows).tail(20)
print("video:",VIDEO_PATH)
print("log:",LOG_PATH)
print("csv:",CSV_PATH)


## Interpretation

This measures whether the generic Laya checkpoint can serve as a fast local zero-shot decision layer for a racing-control state. A useful next step is to collect state -> best_maneuver examples from a competent CarRacing controller, fine-tune/calibrate Laya for this exact 11-action schema, and compare fixed seeds using reward, completion rate, latency, calibration, and fallback count.
